# Prompt 29: Spark Connect — Architecture and Usage
## Databricks Certified Associate Developer for Apache Spark
### Topic 6 — Spark Connect

---

## What is Spark Connect?

Spark Connect is a **new client-server architecture** introduced in **Apache Spark 3.4** (stable in Spark 3.4+, available as preview in 3.3). It decouples the **client-side Spark API** from the **Spark cluster**, creating a **thin client model**.

```
CLASSIC ARCHITECTURE (pre-3.4):
┌─────────────────────────────────────────────┐
│  Application Process (JVM or Python)        │
│  ┌───────────────────────────────────────┐  │
│  │  SparkSession                         │  │
│  │  SparkContext ◄─── Driver lives HERE  │  │
│  │  DAGScheduler                         │  │
│  │  TaskScheduler                        │  │
│  └───────────────────────────────────────┘  │
│  Python Process ◄──► JVM (Py4J bridge)      │
└─────────────────────────────────────────────┘
              │ cluster communication
              ▼
       Spark Cluster (Executors)

SPARK CONNECT ARCHITECTURE (3.4+):
┌────────────────────────┐     gRPC / Protobuf     ┌──────────────────────────────────┐
│  Thin Client Process   │ ◄──────────────────────► │  Spark Connect Server            │
│  (Python / Scala / R)  │    logical plan (proto)  │  ┌────────────────────────────┐  │
│                        │                          │  │  Driver (SparkContext)      │  │
│  SparkSession.remote() │    result (Arrow/Protobuf)│  │  DAGScheduler              │  │
│  DataFrame API         │ ◄──────────────────────  │  │  TaskScheduler             │  │
│  Spark SQL             │                          │  └────────────────────────────┘  │
└────────────────────────┘                          │  Executors                       │
                                                    └──────────────────────────────────┘
```

---

## Classic SparkContext vs Spark Connect

| Dimension | Classic SparkContext | Spark Connect |
|-----------|---------------------|---------------|
| **Driver location** | Embedded in the application process | Runs on the cluster (server-side) |
| **Client process** | Must be a JVM (or Python with Py4J bridge) | Any language with a gRPC stub |
| **Stability** | Client crash kills the driver → job lost | Client crash does NOT kill the driver |
| **Client Spark version** | Must match cluster Spark version exactly | Can differ (protocol negotiation) |
| **Network protocol** | Internal Spark protocol / Py4J socket | gRPC over HTTP/2 + Protocol Buffers |
| **SparkContext access** | Available in client code | **NOT available** on client side |
| **RDD API** | Available | **NOT supported** via Spark Connect |
| **DataFrame API** | Fully available | Fully available |
| **Spark SQL** | Fully available | Fully available |
| **UDFs** | Fully available | Fully available |
| **Installation requirement** | Full Spark installation on client | Only `pyspark` package (thin) |

---

## Protocol: gRPC + Protocol Buffers

Spark Connect uses two key technologies:

**gRPC (Google Remote Procedure Call):**
- A high-performance, open-source RPC framework
- Runs over HTTP/2 — supports streaming, multiplexing, compression
- Used as the transport layer between client and server

**Protocol Buffers (Protobuf):**
- Language-agnostic binary serialization format by Google
- The **logical plan** (query tree) is serialized as Protobuf and sent to the server
- The server deserializes the plan, executes it, and returns results
- Results are returned as **Apache Arrow** (columnar format) for efficient data transfer

**Communication flow:**
```
1. Client builds a DataFrame transformation (lazy — no execution yet)
2. When an action is called (show(), collect(), count()), client serializes
   the logical plan as Protobuf
3. Client sends Protobuf plan to Spark Connect Server via gRPC
4. Server deserializes plan → optimises (Catalyst) → executes
5. Results returned as Apache Arrow batches via gRPC streaming
6. Client deserializes Arrow batches → Python objects
```

---

## Benefits of Spark Connect

1. **Language-agnostic clients** — Any language with a gRPC stub can connect to Spark (Python, Scala, Java, R, Go, etc.) without needing a full Spark installation.

2. **Independent client/server versioning** — The client and server can run different Spark versions (within protocol compatibility). Teams can upgrade the cluster without immediately updating every client application.

3. **Improved stability** — If the client process crashes or is killed, the Spark driver on the server continues running. Jobs are not interrupted.

4. **Works from anywhere** — IDEs, notebooks, microservices, serverless functions, Docker containers — all without a full Spark installation. Only `pip install pyspark` needed.

5. **Thinner client** — No Py4J bridge needed. No JVM running in the client process. Lower memory and startup overhead on the client.

6. **Better multi-tenancy** — Multiple client sessions can share a single Spark Connect server, each with isolated session state.

---

## Starting a Spark Connect Server

```bash
# From SPARK_HOME on the cluster node
./sbin/start-connect-server.sh

# With explicit port (default: 15002)
./sbin/start-connect-server.sh --conf spark.connect.grpc.binding.port=15002

# With additional Spark config
./sbin/start-connect-server.sh \
    --master spark://host:7077 \
    --conf spark.executor.memory=4g \
    --conf spark.executor.cores=2

# Stop
./sbin/stop-connect-server.sh

# Verify it's running (default port 15002)
netstat -tlnp | grep 15002
```

The Spark Connect server is essentially a **long-running Spark application** that listens for gRPC connections on port 15002.

---

## Connecting from Python

```python
from pyspark.sql import SparkSession

# Connect to a remote Spark Connect server
# Protocol: sc:// (Spark Connect), NOT spark:// (Standalone) or local
spark = SparkSession.builder \
    .remote('sc://localhost:15002') \
    .getOrCreate()

# With session name (for identification in server UI)
spark = SparkSession.builder \
    .remote('sc://myserver.example.com:15002') \
    .appName('MyAnalyticsApp') \
    .getOrCreate()

# Protocol prefix MUST be sc:// — NOT spark://, NOT local
# Wrong: .remote('spark://localhost:15002')   # ← connects to Standalone master, not Connect
# Wrong: .master('sc://localhost:15002')      # ← use .remote(), not .master()
```

**Key syntax rule:** `.remote('sc://host:port')` — the `sc://` protocol prefix identifies this as a Spark Connect endpoint.

---

## Supported and Unsupported Operations

**Fully supported via Spark Connect:**
- All standard DataFrame API (`select`, `filter`, `groupBy`, `join`, `withColumn`, etc.)
- Spark SQL (`spark.sql('SELECT ...')`)
- Catalog operations (`spark.catalog.listTables()`, `createOrReplaceTempView()`)
- UDFs (Python and Scala)
- Reading/writing data sources (CSV, JSON, Parquet, Delta, etc.)
- Streaming (via Spark Connect streaming API)

**NOT supported via Spark Connect (exam critical):**
- `SparkContext` — not accessible on the client side
- **RDD API** — no `.rdd`, no `sc.parallelize()`, no `sc.textFile()`
- Accumulators (SparkContext-based)
- Broadcast variables (SparkContext-based, though DataFrame broadcast joins work)
- `spark._jvm` (Py4J JVM access)
- `spark.sparkContext` — accessing this raises an error

```python
# WILL RAISE an error with Spark Connect:
sc = spark.sparkContext         # AnalysisException
rdd = sc.parallelize([1, 2, 3]) # AnalysisException
acc = sc.accumulator(0)          # AnalysisException

# All of these WORK fine:
df = spark.read.csv('/data/file.csv', header=True)
df.groupBy('category').count().show()
spark.sql('SELECT * FROM my_table').show()
```

In [ ]:
# Cell 1: Setup — classic SparkSession (for local demonstration)
# NOTE: The .remote() API requires a running Spark Connect server.
# In this notebook we demonstrate equivalent DataFrame operations using a
# local SparkSession — the API is IDENTICAL to what you would write with
# SparkSession.builder.remote('sc://host:15002').getOrCreate()
# The key exam points are the connection syntax and the supported/unsupported API.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as _sum, desc, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName('SparkConnectDemo') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '2') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

print('=== SparkSession created ===')
print(f'Spark version: {spark.version}')
print()
print('=== Spark Connect connection syntax (for reference) ===')
print('# Production Spark Connect connection:')
print("# spark = SparkSession.builder.remote('sc://myserver.example.com:15002').getOrCreate()")
print()
print('# Protocol is sc:// NOT spark://')
print('# Use .remote() NOT .master()')
print('# Default port: 15002')

In [ ]:
# Cell 2: DataFrame operations that work identically with Spark Connect
# The DataFrame API is the SAME whether using classic SparkSession or .remote()

# Sample sales data
sales_data = [
    ('2026-01', 'Electronics', 'Laptop',    1499.99, 3),
    ('2026-01', 'Electronics', 'Phone',      799.99, 8),
    ('2026-01', 'Clothing',    'Jacket',     129.99, 15),
    ('2026-01', 'Clothing',    'Shoes',       89.99, 22),
    ('2026-01', 'Books',       'Python Book', 39.99, 12),
    ('2026-02', 'Electronics', 'Laptop',    1499.99, 5),
    ('2026-02', 'Electronics', 'Tablet',     599.99, 4),
    ('2026-02', 'Clothing',    'Jacket',     129.99, 9),
    ('2026-02', 'Books',       'Spark Book',  44.99, 7),
    ('2026-02', 'Books',       'Python Book', 39.99, 20),
]

schema = StructType([
    StructField('month',    StringType(),  True),
    StructField('category', StringType(),  True),
    StructField('product',  StringType(),  True),
    StructField('price',    DoubleType(),  True),
    StructField('quantity', IntegerType(), True),
])

df = spark.createDataFrame(sales_data, schema) \
    .withColumn('revenue', col('price') * col('quantity'))

print('=== Sales data ===')
df.show()

# -------------------------------------------------------
# 1. groupBy aggregation — identical API with Spark Connect
# -------------------------------------------------------
print('=== Revenue by category (groupBy + agg) ===')
df.groupBy('category') \
  .agg(
      count('product').alias('num_products'),
      _sum('quantity').alias('total_units'),
      _sum('revenue').alias('total_revenue'),
      avg('price').alias('avg_price')
  ) \
  .orderBy(desc('total_revenue')) \
  .show()

# -------------------------------------------------------
# 2. Spark SQL — identical with Spark Connect
# -------------------------------------------------------
df.createOrReplaceTempView('sales')

print('=== Top products by revenue (Spark SQL) ===')
spark.sql("""
    SELECT product, category,
           SUM(revenue) as total_revenue,
           SUM(quantity) as total_units
    FROM sales
    GROUP BY product, category
    ORDER BY total_revenue DESC
    LIMIT 5
""").show()

In [ ]:
# Cell 3: UDFs — supported with Spark Connect
# UDFs registered via spark.udf.register() work identically

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Python UDF — works with Spark Connect
def categorise_revenue(revenue):
    if revenue is None:
        return 'Unknown'
    elif revenue >= 5000:
        return 'High'
    elif revenue >= 1000:
        return 'Medium'
    else:
        return 'Low'

# Register via udf() decorator — works with Spark Connect
categorise_revenue_udf = udf(categorise_revenue, StringType())

print('=== UDF example — revenue tier per row ===')
df.withColumn('revenue_tier', categorise_revenue_udf(col('revenue'))) \
  .select('product', 'category', 'revenue', 'revenue_tier') \
  .orderBy(desc('revenue')) \
  .show()

# Also works via SQL UDF registration — works with Spark Connect
spark.udf.register('categorise_rev', categorise_revenue, StringType())
print('=== SQL UDF example ===')
spark.sql("""
    SELECT product, revenue, categorise_rev(revenue) as tier
    FROM sales
    ORDER BY revenue DESC
    LIMIT 5
""").show()

In [ ]:
# Cell 4: What does NOT work with Spark Connect — SparkContext and RDD API

print('=== Operations NOT supported via Spark Connect ===')
print()
print('The following would raise errors when using SparkSession.builder.remote():')
print()

# Demonstrate by catching the errors in classic mode when we try to use
# SparkContext in an isolated way — and explain the Spark Connect restriction

unsupported = [
    ("spark.sparkContext",
     "SparkContext is not accessible via Spark Connect client. "
     "The driver runs on the server — the client has no SparkContext."),

    ("sc.parallelize([1, 2, 3])",
     "RDD API requires SparkContext — not available on Spark Connect client."),

    ("sc.textFile('/path/to/file.txt')",
     "RDD-based file reads require SparkContext."),

    ("sc.accumulator(0)",
     "Accumulators require SparkContext — not available on Spark Connect client."),

    ("sc.broadcast(large_dict)",
     "SparkContext broadcast variables not available. "
     "Use DataFrame broadcast joins (functions.broadcast()) instead."),

    ("spark._jvm.SomeClass()",
     "Direct Py4J JVM access not available — no JVM in the thin client process."),
]

print(f'{"Operation":<35} {"Reason"}')
print('-' * 95)
for op, reason in unsupported:
    print(f'{op:<35} {reason}')

print()
print('=== What to use instead ===')
alternatives = [
    ('sc.parallelize(data)',          'spark.createDataFrame(data, schema)'),
    ('sc.textFile(path)',             'spark.read.text(path)'),
    ('sc.broadcast(dict)',            'df.join(broadcast(lookup_df), ...)'),
    ('rdd.map(...)',                  'df.select(expr(...)) or df.withColumn(...)'),
    ('rdd.flatMap(...)',              'df.select(explode(...))'),
    ('rdd.filter(...)',               'df.filter(condition)'),
]
print(f'{"Instead of":<35} {"Use"}')
print('-' * 70)
for old, new in alternatives:
    print(f'{old:<35} {new}')

In [ ]:
# Cell 5: Complete Spark Connect workflow simulation
# Shows the full pattern: connect → read → transform → write
# This is exactly what you'd write with .remote() pointing at a real server

import os, tempfile

# Step 1: Create a CSV data source (simulating files on HDFS/cloud storage)
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, 'employees.csv')
output_path = os.path.join(tmp_dir, 'dept_summary')

with open(csv_path, 'w') as f:
    f.write('emp_id,name,department,salary,years\n')
    f.write('1,Alice,Engineering,95000,5\n')
    f.write('2,Bob,Engineering,85000,3\n')
    f.write('3,Carol,Marketing,72000,7\n')
    f.write('4,Dave,Marketing,68000,2\n')
    f.write('5,Eve,Engineering,105000,8\n')
    f.write('6,Frank,HR,62000,4\n')
    f.write('7,Grace,HR,58000,1\n')
    f.write('8,Hank,Marketing,75000,6\n')

print('=== Complete Spark Connect workflow ===')
print()
print('# Step 1: Connect (in real Spark Connect scenario):')
print("# spark = SparkSession.builder.remote('sc://myserver:15002').getOrCreate()")
print()

# Step 2: Read CSV — identical to .remote() session
print('# Step 2: Read CSV')
df_emp = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(csv_path)

print('Schema:')
df_emp.printSchema()

# Step 3: Transform — groupBy aggregation
print('# Step 3: GroupBy aggregation by department')
df_summary = df_emp \
    .groupBy('department') \
    .agg(
        count('emp_id').alias('headcount'),
        avg('salary').alias('avg_salary'),
        _sum('salary').alias('total_salary_cost'),
        avg('years').alias('avg_tenure')
    ) \
    .orderBy(desc('total_salary_cost'))

print('\nDepartment summary:')
df_summary.show()

# Step 4: Write results — identical to .remote() session
print('# Step 4: Write results as Parquet')
df_summary.write \
    .mode('overwrite') \
    .parquet(output_path)
print(f'Written to: {output_path}')

# Step 5: Verify by reading back
df_verify = spark.read.parquet(output_path)
print('\n# Verification — read back from Parquet:')
df_verify.show()

spark.stop()
print('\nSession closed.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Data scientist runs Spark queries from a Jupyter notebook on a laptop — no Spark installed</summary>

**Situation:**
A data scientist wants to analyse 500 GB of sales data using PySpark from their laptop. Installing a full Spark cluster locally is impractical. The company has a shared Spark Connect server running on a cluster.

**Solution — before Spark Connect:**
```bash
# Had to install Spark locally, set JAVA_HOME, configure spark-defaults.conf...
# OR use a remote YARN cluster — but then the driver runs on the laptop,
# must stay connected, and requires a complex network setup
```

**Solution — with Spark Connect:**
```bash
# On the laptop — just pip install:
pip install pyspark
```

```python
# In Jupyter notebook:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, desc

spark = SparkSession.builder \
    .remote('sc://spark-server.company.com:15002') \
    .appName('SalesAnalysis') \
    .getOrCreate()

df = spark.read.parquet('/data/warehouse/sales/')
top_products = df.groupBy('product_id') \
    .agg(_sum('revenue').alias('total_revenue')) \
    .orderBy(desc('total_revenue')) \
    .limit(10)

top_products.show()
# All computation runs on the cluster — laptop only receives the 10-row result
```

**Benefits demonstrated:** Language-agnostic thin client; no full Spark install on laptop; only result rows transferred to client.

**Exam Sub-topic:** Spark Connect benefits; `.remote()` syntax; thin client model
</details>

<details>
<summary>Scenario 2: Developer migrates code from RDD API to DataFrame API to support Spark Connect</summary>

**Situation:**
A developer wants to modernise a legacy application to use Spark Connect. The existing code uses the RDD API which is not supported via Spark Connect.

**Legacy code (RDD API — not compatible with Spark Connect):**
```python
sc = spark.sparkContext  # ← AnalysisException with Spark Connect

rdd = sc.textFile('/data/sales.csv')  # ← not supported
header = rdd.first()
data_rdd = rdd.filter(lambda line: line != header)

revenue_rdd = data_rdd \
    .map(lambda line: line.split(',')) \
    .map(lambda fields: (fields[2], float(fields[3]) * int(fields[4])))

revenue_by_cat = revenue_rdd.reduceByKey(lambda a, b: a + b)
revenue_by_cat.take(5)
```

**Migrated code (DataFrame API — fully compatible with Spark Connect):**
```python
# spark = SparkSession.builder.remote('sc://server:15002').getOrCreate()

df = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv('/data/sales.csv')

revenue_by_cat = df \
    .withColumn('revenue', col('price') * col('quantity')) \
    .groupBy('category') \
    .agg(_sum('revenue').alias('total_revenue')) \
    .orderBy(desc('total_revenue'))

revenue_by_cat.show(5)  # Works perfectly with Spark Connect
```

**Exam Sub-topic:** RDD API not supported via Spark Connect; DataFrame API is the replacement; `SparkContext` not accessible client-side
</details>

<details>
<summary>Scenario 3: Comparing what happens when a client crashes under classic vs Spark Connect</summary>

**Situation:**
A Spark job is processing 2 TB of data (expected duration: 4 hours). The client machine loses network connectivity at the 2-hour mark.

**Classic SparkSession (driver in client process):**
```
Client machine hosts the Driver → network loss → Driver loses connection to executors
→ Job FAILS → all 2 hours of work lost → must restart from scratch
```

**Spark Connect:**
```
Client machine is thin → network loss → client loses connection to the server
→ Driver on server is UNAFFECTED → job continues running
→ When client reconnects: spark.reconnect() to re-attach to the running query
→ 2 hours of work preserved
```

```python
# Spark Connect: long-running job survives client disconnection
spark = SparkSession.builder.remote('sc://server:15002').getOrCreate()

# Start a long job
query = df_large.writeStream \
    .format('delta') \
    .outputMode('append') \
    .option('checkpointLocation', '/ckpt/big-job') \
    .start('/data/output')

# ... client disconnects ...
# ... client reconnects later ...

# Re-attach to the same server session (session ID preserved)
spark2 = SparkSession.builder.remote('sc://server:15002').getOrCreate()
```

**Exam Sub-topic:** Improved stability with Spark Connect; client crash vs driver crash; driver runs on server
</details>

<details>
<summary>Scenario 4: A microservice calls Spark for real-time aggregation via Spark Connect</summary>

**Situation:**
A Python FastAPI microservice needs to query a Spark cluster for on-demand aggregations when serving API requests. The microservice runs in a Docker container with no Spark installation.

**Solution:**
```dockerfile
# Dockerfile — only pyspark needed, no Spark installation
FROM python:3.11-slim
RUN pip install fastapi uvicorn pyspark
# No Java, no Spark binaries needed for Spark Connect client
```

```python
# app.py — FastAPI microservice using Spark Connect
from fastapi import FastAPI
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum

app = FastAPI()

# Shared SparkSession — connected once at startup
spark = SparkSession.builder \
    .remote('sc://spark-server.internal:15002') \
    .appName('ReportingAPI') \
    .getOrCreate()

@app.get('/revenue/{department}')
def get_dept_revenue(department: str):
    result = spark.read.parquet('/data/sales') \
        .filter(col('department') == department) \
        .agg(_sum('revenue').alias('total_revenue')) \
        .collect()
    return {'department': department, 'total_revenue': result[0]['total_revenue']}
```

**Exam Sub-topic:** Works from microservices/Docker without full Spark install; thin client model; language-agnostic architecture
</details>

<details>
<summary>Scenario 5: Team upgrades the Spark cluster without breaking existing client applications</summary>

**Situation:**
A team wants to upgrade their Spark cluster from 3.4 to 3.5 without forcing all 12 client teams to simultaneously update their `pyspark` package version.

**Classic approach (without Spark Connect):**
```
Problem: pyspark client version must match the cluster Spark version exactly.
Upgrading the cluster from 3.4 → 3.5 breaks all clients still on pyspark==3.4.*
Solution: All teams must coordinate a simultaneous upgrade → high coordination cost
```

**With Spark Connect:**
```
Client pyspark version and cluster Spark version are DECOUPLED via the Protobuf protocol.
The server advertises its protocol version; the client negotiates compatibility.

Step 1: Upgrade cluster to Spark 3.5 (with Spark Connect server)
Step 2: Existing clients on pyspark 3.4 continue to work
Step 3: Client teams upgrade pyspark on their own schedule
```

```bash
# Server team upgrades the cluster:
# ./sbin/stop-connect-server.sh
# [upgrade Spark binaries to 3.5]
# ./sbin/start-connect-server.sh

# Client teams — no immediate change needed:
pip show pyspark  # still shows 3.4.x — works fine with 3.5 server

# Client teams upgrade when ready:
pip install --upgrade pyspark  # upgrade to 3.5.x at their own pace
```

**Exam Sub-topic:** Independent client/server versioning; decoupled deployment; key benefit of Spark Connect
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| What is Spark Connect? | Client-server architecture decoupling client API from cluster (Spark 3.4+) |
| Introduced in Spark version | **3.4** (stable release) |
| Network protocol | **gRPC** over HTTP/2 |
| Serialisation format for plans | **Protocol Buffers (Protobuf)** |
| Result transfer format | **Apache Arrow** (columnar) |
| Default port | **15002** |
| Start server command | `./sbin/start-connect-server.sh` |
| Stop server command | `./sbin/stop-connect-server.sh` |
| Python connection syntax | `SparkSession.builder.remote('sc://host:15002').getOrCreate()` |
| Protocol prefix | `sc://` (not `spark://`, not `local`) |
| Method name | `.remote()` (not `.master()`) |
| Where does the driver run? | **On the cluster** (server-side), not in the client process |
| Is `SparkContext` available client-side? | **No** — raises `AnalysisException` |
| Is the RDD API supported? | **No** — not supported via Spark Connect |
| Is DataFrame API supported? | **Yes** — fully supported |
| Is Spark SQL supported? | **Yes** — fully supported |
| Are UDFs supported? | **Yes** — fully supported |
| Client crash effect on job | Job **continues** — driver on server unaffected |
| Classic SparkContext crash effect | Job **fails** — driver in client process |
| Client/server version matching | **Not required** — independently versioned |
| Spark installation needed on client? | **No** — only `pip install pyspark` |

---

## Practice Q&A

<details>
<summary>Q1: What is the key architectural difference between a classic SparkSession and a Spark Connect SparkSession?</summary>

**A:**

| | Classic SparkSession | Spark Connect SparkSession |
|-|---------------------|----------------------------|
| **Driver location** | **In the client/application process** | **On the cluster** (Spark Connect server) |
| **Communication** | Internal Spark protocol / Py4J | gRPC + Protobuf over HTTP/2 |
| **Client crash** | Kills the driver and the job | Job continues — driver unaffected |
| **SparkContext** | Accessible in client code | NOT accessible |
| **RDD API** | Available | NOT available |

In the classic model, the SparkContext (Driver) is embedded in the client process. The client IS the driver. In Spark Connect, the client is a thin process that sends logical plans to a remote driver over gRPC.
</details>

<details>
<summary>Q2: How do you connect to a Spark Connect server running on `spark-cluster.example.com` at the default port from Python?</summary>

**A:**
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .remote('sc://spark-cluster.example.com:15002') \
    .getOrCreate()
```

**Key exam points:**
- Use `.remote('sc://...')` — NOT `.master('sc://...')` (`.master()` is for Standalone/YARN/etc.)
- Protocol prefix is `sc://` — NOT `spark://` (that's Standalone cluster manager)
- Default port: **15002**
- No other configuration needed — the server handles all cluster resource management
</details>

<details>
<summary>Q3: A developer writes `sc = spark.sparkContext` in their Spark Connect application. What happens?</summary>

**A:** The code raises an `AnalysisException` (or `SparkConnectException` in newer versions).

With Spark Connect, the client is a **thin process** that communicates with the Spark driver over gRPC. The SparkContext (which is the driver) lives on the server — it is not accessible from the client code.

Any attempt to access `SparkContext` on the client will fail:
```python
sc = spark.sparkContext  # ← exception raised
```

**Alternatives for common SparkContext operations:**
- `sc.parallelize(data)` → `spark.createDataFrame(data, schema)`
- `sc.textFile(path)` → `spark.read.text(path)`
- `sc.broadcast(dict)` → DataFrame broadcast join with `functions.broadcast()`
</details>

<details>
<summary>Q4: What network protocol and serialisation format does Spark Connect use, and why were they chosen?</summary>

**A:**

**Protocol: gRPC over HTTP/2**
- High-performance RPC framework by Google
- HTTP/2 supports multiplexing, streaming, and compression
- Language-agnostic: gRPC stubs exist for Python, Java, Go, C++, Ruby, etc.
- Chosen because it enables non-JVM clients to connect to Spark

**Serialisation: Protocol Buffers (Protobuf)**
- Language-agnostic binary serialisation format (more compact than JSON)
- The client's logical plan (query tree) is serialised as Protobuf and sent to the server
- The server deserialises → optimises with Catalyst → executes

**Results format: Apache Arrow**
- Columnar in-memory format
- Results are streamed back from server to client as Arrow record batches
- Very efficient for numeric and structured data

**Why these choices:** Together they enable any language that has gRPC support to act as a Spark client, without needing a JVM or a full Spark installation.
</details>

<details>
<summary>Q5: List three benefits of Spark Connect over the classic SparkSession architecture.</summary>

**A:** (Any three from the following)

1. **Language-agnostic clients** — Any language with a gRPC stub can connect to Spark. You are not limited to JVM languages or Python-with-Py4J.

2. **Independent client/server versioning** — The client `pyspark` version and the cluster Spark version do not need to match. Teams can upgrade independently.

3. **Improved stability** — If the client process crashes or disconnects, the Spark driver on the server continues running and the job is not interrupted. In the classic model, a client crash kills the driver.

4. **Works from anywhere without full Spark installation** — IDEs, notebooks, Docker containers, microservices, serverless functions — only `pip install pyspark` is needed (no Java, no Spark binaries on client).

5. **Better multi-tenancy** — Multiple isolated client sessions can share a single Spark Connect server, each with their own session state.
</details>